# Part B: Reproduce the Same RNN in PyTorch

In [ ]:
import torch                  # библиотека для тензоров и DL
import torch.nn as nn        # нейросетевые модули

# создаем входной тензор
# форма: (batch_size=1, seq_len=4, input_size=3)
x = torch.tensor([[
    [1.0, 0.0, 0.0],   # x1
    [0.0, 1.0, 0.0],   # x2
    [0.0, 0.0, 1.0],   # x3
    [1.0, 1.0, 0.0],   # x4
]])

# создаем RNN слой
rnn = nn.RNN(
    input_size=3,      # размер входного вектора (x_t)
    hidden_size=2,     # размер hidden state (h_t)
    batch_first=True   # формат (batch, seq, features)
)

# прогоняем данные через RNN
output, h_n = rnn(x)

# выводим формы тензоров
print("x.shape:", x.shape)            # (1, 4, 3) 1 последовательность, 4 шага, 3 признака
print("output.shape:", output.shape)  # (1, 4, 2) на каждом шаге есть hidden state размера 2
print("h_n.shape:", h_n.shape)        # (1, 1, 2) финальный hidden state

# выводим значения (для понимания)
print("\noutput:\n", output) #последний hidden state из output
print("\nh_n:\n", h_n) #финальный hidden state

x.shape: torch.Size([1, 4, 3])
output.shape: torch.Size([1, 4, 2])
h_n.shape: torch.Size([1, 1, 2])

output:
 tensor([[[-0.0501, -0.0961],
         [-0.0729, -0.6098],
         [ 0.6424, -0.0102],
         [-0.5217, -0.0882]]], grad_fn=<TransposeBackward1>)

h_n:
 tensor([[[-0.5217, -0.0882]]], grad_fn=<StackBackward0>)


# Part C: DNA Sequence Classifier

## C1. Generate Data

In [ ]:
import random  # для случайных чисел

alphabet = ["A", "C", "G", "T"]  # возможные буквы
motif = "ATG"  # что ищем

# перевод букв в числа
char2idx = {"A":0, "C":1, "G":2, "T":3, "<PAD>":4}

# создаём одну строку
def generate_sequence(min_len=30, max_len=80, positive=True):
    length = random.randint(min_len, max_len)  # случайная длина

    # случайная строка
    seq = "".join(random.choice(alphabet) for _ in range(length))

    if positive:
        # вставляем ATG
        pos = random.randint(0, length - 3)
        seq = seq[:pos] + "ATG" + seq[pos+3:]
    else:
        # убираем ATG
        while "ATG" in seq:
            seq = "".join(random.choice(alphabet) for _ in range(length))

    return seq

# создаём датасет
def build_dataset(n):
    X, y = [], []
    for _ in range(n):
        label = random.choice([0,1])  # 0 или 1
        seq = generate_sequence(positive=bool(label))
        X.append(seq)
        y.append(label)
    return X, y

train_X, train_y = build_dataset(2000)
val_X, val_y = build_dataset(500)
test_X, test_y = build_dataset(500)

# посмотрим
for i in range(3):
    print(train_X[i][:40], "... →", train_y[i])

TGTAGTTGGGCCGCCTTACTATGCAATTCTTAATAGGAAT ... → 1
TGGTCTCGCAGATACCACACCCCTAATTACCCTTCAAGCC ... → 0
CATTCCATCTCGATGATGTTAGGCCATGTTACAACAACAC ... → 1


## C2. Encode and Pad

In [ ]:
import torch
import torch.nn as nn

# перевод строки → числа
def encode(seq):
    return [char2idx[c] for c in seq]

# делаем batch
def collate(batch):
    sequences, labels = zip(*batch)

    # кодируем
    encoded = [torch.tensor(encode(s)) for s in sequences]

    # длины
    lengths = torch.tensor([len(s) for s in encoded])

    # добавляем PAD (чтобы длины были одинаковые)
    padded = nn.utils.rnn.pad_sequence(
        encoded,
        batch_first=True,
        padding_value=4
    )

    return padded, lengths, torch.tensor(labels)

# проверка
sample = list(zip(train_X[:3], train_y[:3]))
x_batch, lengths, y_batch = collate(sample)

print("x_batch.shape:", x_batch.shape)
print("lengths:", lengths)
print("y_batch:", y_batch)

x_batch.shape: torch.Size([3, 76])
lengths: tensor([70, 69, 76])
y_batch: tensor([1, 0, 1])


все строки стали одинаковой длины
lengths = настоящая длина

## C3. Train a Classifier

In [ ]:
class DNAClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(5, 8, padding_idx=4)
        # 5 символов → вектор размера 8

        self.rnn = nn.RNN(8, 16, batch_first=True)
        # input=8, hidden=16

        self.fc = nn.Linear(16, 2)
        # выход → 2 класса

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        print("embedded.shape:", embedded.shape)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, h_n = self.rnn(packed)
        print("h_n.shape:", h_n.shape)

        last_hidden = h_n[-1]
        print("last_hidden:", last_hidden.shape)

        out = self.fc(last_hidden)
        print("logits:", out)

        return out

model = DNAClassifier()

# тест
_ = model(x_batch, lengths)

embedded.shape: torch.Size([3, 76, 8])
h_n.shape: torch.Size([1, 3, 16])
last_hidden: torch.Size([3, 16])
logits: tensor([[0.1058, 0.6923],
        [0.1099, 0.5985],
        [0.3491, 0.2412]], grad_fn=<AddmmBackward0>)


embedding → превращает числа в вектора
RNN → читает
last_hidden → память
logits → ответ

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(list(zip(train_X, train_y)), batch_size=32, shuffle=True, collate_fn=collate)
val_loader = DataLoader(list(zip(val_X, val_y)), batch_size=32, collate_fn=collate)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate(loader):
    correct, total = 0, 0
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x, lengths)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(3):
    for x, lengths, y in train_loader:
        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    print("Epoch", epoch+1,
          "Train:", evaluate(train_loader),
          "Val:", evaluate(val_loader))

Выходные данные были обрезаны до нескольких последних строк (5000).
        [0.2352, 0.3459],
        [0.1879, 0.3532],
        [0.1528, 0.3608],
        [0.2553, 0.0758],
        [0.1585, 0.3757],
        [0.1431, 0.2934],
        [0.0861, 0.4211],
        [0.3381, 0.3727],
        [0.2880, 0.3806],
        [0.1415, 0.3996],
        [0.2694, 0.1506],
        [0.1633, 0.4163],
        [0.1740, 0.4869],
        [0.2071, 0.0549]])
embedded.shape: torch.Size([32, 79, 8])
h_n.shape: torch.Size([1, 32, 16])
last_hidden: torch.Size([32, 16])
logits: tensor([[0.2575, 0.4464],
        [0.1940, 0.3318],
        [0.1566, 0.4098],
        [0.2261, 0.3905],
        [0.2256, 0.3663],
        [0.3343, 0.3167],
        [0.2946, 0.3916],
        [0.3711, 0.2786],
        [0.1705, 0.4045],
        [0.1468, 0.4472],
        [0.0546, 0.4526],
        [0.0801, 0.2634],
        [0.1611, 0.4344],
        [0.2921, 0.3083],
        [0.1818, 0.3417],
        [0.2309, 0.4066],
        [0.1689, 0.3964],
        

## C4. Generalization Tests

In [ ]:
def predict(seq):
    x = torch.tensor([encode(seq)])
    lengths = torch.tensor([len(seq)])

    with torch.no_grad():
        logits = model(x, lengths)
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(dim=1).item()
        prob = probs[0,1].item()

    return pred, prob

tests = [
    "ATGAAAAAAA",
    "AAAAAATGAAAA",
    "GGGGGGGGGG"
]

for t in tests:
    pred, prob = predict(t)
    print(t, "→", pred, "prob:", round(prob,2))

embedded.shape: torch.Size([1, 10, 8])
h_n.shape: torch.Size([1, 1, 16])
last_hidden: torch.Size([1, 16])
logits: tensor([[0.3749, 0.1816]])
ATGAAAAAAA → 0 prob: 0.45
embedded.shape: torch.Size([1, 12, 8])
h_n.shape: torch.Size([1, 1, 16])
last_hidden: torch.Size([1, 16])
logits: tensor([[0.3682, 0.2041]])
AAAAAATGAAAA → 0 prob: 0.46
embedded.shape: torch.Size([1, 10, 8])
h_n.shape: torch.Size([1, 1, 16])
last_hidden: torch.Size([1, 16])
logits: tensor([[0.0110, 0.3356]])
GGGGGGGGGG → 1 prob: 0.58


## Follow-Up Questions for Understanding

### 1. Почему мы используем `nn.Embedding` перед RNN?

`nn.Embedding` нужен для преобразования номеров токенов (индексов) в плотные векторы признаков.

RNN не понимает сами числа вроде 1, 2, 3, потому что это просто индексы, а не смысловая информация.

Embedding помогает модели обучать представления, где похожие элементы могут иметь похожие векторы.

Например, вместо того чтобы подавать ID символа напрямую, мы подаем его векторное представление.

---

### 2. Почему это задача many-to-one?

Это many-to-one задача, потому что на входе у нас целая последовательность из многих элементов, а на выходе только одна метка.

Пример:

Вход: DNA-последовательность → A C G T A T G C  
Выход: 1 (мотив есть) или 0 (мотива нет)

Много входных шагов → один итоговый ответ.

---

### 3. Почему для классификации мы используем последнее скрытое состояние?

Последнее hidden state содержит информацию обо всей последовательности, потому что RNN читает элементы по одному и обновляет своё состояние на каждом шаге.

К последнему шагу hidden state должен хранить самое важное из всей последовательности.

Поэтому именно его используют для финальной классификации.

Это как сначала дочитать весь текст, а потом отвечать на вопрос.

---

### 4. Почему мы делаем padding последовательностей в batch?

Последовательности обычно имеют разную длину, но в одном batch все тензоры должны быть одинаковой формы.

Padding добавляет дополнительные нули к коротким последовательностям, чтобы все стали одной длины.

Без этого batch-обучение работать не будет.

Пример:

ATG  
ATGCCGA

становится

ATG0000  
ATGCCGA

Так модель может обрабатывать их одновременно.

---

### 5. Почему `pack_padded_sequence` помогает при разной длине последовательностей?

Без packing RNN обрабатывает даже добавленные нули (padding), а это лишняя работа и может ухудшать качество модели.

`pack_padded_sequence` сообщает PyTorch, где настоящие данные, а где просто padding.

Это ускоряет обучение и помогает модели сосредоточиться только на полезной информации.

Короче: меньше работы с мусором, больше пользы.

---

### 6. Какую информацию vanilla RNN может забывать на длинных последовательностях?

Обычная RNN может забывать информацию из начала длинной последовательности.

Это происходит из-за проблемы vanishing gradient (затухающего градиента).

Старые данные становятся всё слабее после большого количества шагов.

Например, если важный мотив был в начале DNA-последовательности, к концу RNN может его уже “забыть”.

Типичная RNN: «Я это знала 20 шагов назад, а сейчас уже не помню».

---

### 7. Какую проблему LSTM и GRU уменьшают по сравнению с обычной RNN?

LSTM и GRU созданы для уменьшения проблемы vanishing gradient.

Они помогают сохранять важную информацию на более длинных промежутках.

Поэтому они лучше работают с длинными последовательностями, где ранняя информация всё ещё важна позже.

Это как RNN, но с нормальной памятью.

---

### 8. Что решают gates (ворота) в LSTM и GRU во время чтения последовательности?

Gates помогают решать:

- какую информацию сохранить  
- какую забыть  
- какую новую информацию добавить

Это позволяет модели лучше управлять своей памятью.

Вместо того чтобы помнить всё подряд или забывать всё подряд, модель выбирает только важное.

Очень взрослая стратегия, если честно.

---

### 9. Если заменить `nn.RNN` на `nn.LSTM`, что дополнительно возвращает PyTorch?

`nn.LSTM` возвращает дополнительное cell state помимо hidden state.

Обычная RNN возвращает:

- output
- hidden state (`h_n`)

LSTM возвращает:

- output
- hidden state (`h_n`)
- cell state (`c_n`)

Cell state помогает хранить долгосрочную память намного лучше.

То есть LSTM буквально говорит: «Я принесла дополнительное хранилище».